# Master Test Suite Runner: run_all_tests
**Purpose:** Executes all Lakehouse testing suites (`01_test_landing`, `02_test_bronze_silver`, `03_test_gold`) in sequence and reports a consolidated pass/fail scorecard.


In [ ]:
%run ../notebooks/00_config

import time

suites = [
    {"name": "01_Landing_Gate_Tests", "path": "./01_test_landing", "timeout": 300},
    {"name": "02_Bronze_Silver_Tests", "path": "./02_test_bronze_silver", "timeout": 600},
    {"name": "03_Gold_Star_Schema_Tests", "path": "./03_test_gold", "timeout": 600}
]

results = []
print("=" * 80)
print(f"STARTING COMPREHENSIVE LAKEHOUSE TEST SUITE: {CATALOG_NAME}")
print("=" * 80)

for suite in suites:
    t0 = time.time()
    print(f"\n--> Running: {suite['name']}...")
    try:
        dbutils.notebook.run(suite['path'], suite['timeout'])
        duration = round(time.time() - t0, 1)
        results.append({"suite": suite['name'], "status": "PASS", "duration": duration, "error": None})
        print(f"    [PASS] {suite['name']} completed in {duration}s")
    except Exception as e:
        duration = round(time.time() - t0, 1)
        results.append({"suite": suite['name'], "status": "FAIL", "duration": duration, "error": str(e)})
        print(f"    [FAIL] {suite['name']} failed in {duration}s: {e}")

print("\n" + "=" * 80)
print("TEST EXECUTION SCORECARD SUMMARY")
print("=" * 80)
total_pass = sum(1 for r in results if r['status'] == 'PASS')
total_fail = sum(1 for r in results if r['status'] == 'FAIL')

for r in results:
    stat = r['status']
    print(f"[{stat}] {r['suite']:<35} Duration: {r['duration']}s")
    if r['error']:
        print(f"       Error: {r['error'][:100]}...")

print("-" * 80)
print(f"Total Suites: {len(results)} | Passed: {total_pass} | Failed: {total_fail}")
print("=" * 80)

if total_fail > 0:
    raise AssertionError(f"Lakehouse testing failed with {total_fail} failed suites.")
else:
    print("ALL LAKEHOUSE ASSERTION TESTS PASSED SUCCESSFULLY!")
